# Google Cloud Text-to-Speech

A refresher on **Google Cloud Text-to-Speech (Cloud TTS)** — Google's hosted speech-synthesis API. You `POST` text (or **SSML**) plus a *voice* and an *audio config*, and get back audio bytes. It exposes Google's voice ladder from cheap **Standard** voices up through **WaveNet**, **Neural2**, **Studio**, and the newest generative **Chirp 3: HD** voices, across ~50 languages. Zero model ops: you hold credentials and call an endpoint; the models run on Google's infrastructure, billed per character.

**Domain:** Speech & Audio  ·  **from study list**  ·  **runnable:** yes  ·  _the no-key cells (cost math + request shape) run on CPU with stdlib only; the real synthesis call is gated behind `GOOGLE_APPLICATION_CREDENTIALS`_

## 1. What & Why

**What it is.** Cloud Text-to-Speech is a REST/gRPC API in Google Cloud Platform that converts text into natural-sounding audio. The core call takes three things: a **`SynthesisInput`** (plain text *or* SSML), a **`VoiceSelectionParams`** (which language + which voice), and an **`AudioConfig`** (encoding, sample rate, speaking rate, pitch). It returns raw audio bytes (MP3, LINEAR16/WAV, OGG_OPUS, µ-law for telephony, …). Synchronous synthesis is capped at **5000 bytes** of input per request; longer documents use the separate **Long Audio API** (`synthesize_long_audio`, async, writes to a Cloud Storage bucket).

**The problem it solves.** Running your own TTS (Coqui, Piper, StyleTTS2) means GPUs, model downloads, vocoder tuning, and quality that still trails the best. Cloud TTS removes all of that: a deep catalog of voices, ~50 languages/380+ voices, SSML for fine prosody control, and enterprise plumbing (IAM, VPC-SC, SLAs, regional endpoints) behind one API call. You trade money and a network round-trip for solid quality with zero ML ops and Google-grade compliance.

**When to reach for it.** Apps already on GCP; IVR/telephony (native µ-law + device-profile audio tuning); accessibility and read-aloud; multilingual products that need one vendor for ~50 languages; batch narration via the Long Audio API. It's the default hosted TTS when you want a wide voice catalog, SSML control, and GCP-native security rather than absolute top-tier naturalness.

**When not to.** When you need the most expressive/emotive voices or instant voice cloning (ElevenLabs leads there); when audio cannot leave your machine (use Piper/Coqui offline); or for very high-volume batch where per-character cost on WaveNet/Studio dominates and a cheaper self-hosted model would do. Note Cloud TTS has **no general voice-cloning** product — you pick from Google's catalog (Custom Voice exists but is gated/enterprise).

## 2. Mental Model

**Cloud TTS is a voice vending machine with a quality dial. You don't own the model — you authenticate with a Google credential, then for every request you choose three orthogonal things: *what to say* (text or SSML), *who says it* (language + voice name, which also picks a quality tier), and *how the audio is shaped* (encoding, rate, pitch). Audio comes out; you're billed per character of input.**

```
                 your machine                                Google Cloud (their infra)
 ┌──────────────────────────────────┐   HTTPS / gRPC   ┌────────────────────────────────────┐
 │ synthesize_speech(                │ ───────────────▶ │  voice tier (chosen by voice name): │
 │   input  = text OR ssml,          │   ADC / API key  │   Standard → WaveNet → Neural2 →    │
 │   voice  = language_code + name,  │                  │   Studio → Chirp 3: HD             │
 │   audio_config = encoding, rate,  │                  │  synthesize on Google TPUs/GPUs    │
 │                  pitch, sample_hz │ ◀─────────────── │  ── audio bytes (mp3/wav/opus/…) ──▶│
 └──────────────────────────────────┘   audio bytes    └────────────────────────────────────┘
        ≤ 5000 bytes/request (else use the async Long Audio API)
```

Three knobs decide everything, and they're independent:

1. **`input` = *what*** — a `text` string, or an `ssml` string (`<speak>…</speak>`) for pauses, emphasis, `<say-as>`, phonemes, etc. **Billing counts every character of the input, SSML tags included.**
2. **`voice` = *who + quality tier*** — `language_code` (e.g. `en-US`) plus a `name` (e.g. `en-US-Neural2-F`). The naming convention `{lang}-{Tier}-{Letter}` *is* the quality/price selector: `Standard`, `Wavenet`, `Neural2`, `Studio`, `Chirp3-HD`. Omit `name` and Google picks a default for the language + `ssml_gender`.
3. **`audio_config` = *how it sounds/encodes*** — `audio_encoding` (MP3, LINEAR16, OGG_OPUS, MULAW, ALAW), `speaking_rate` (0.25–4.0), `pitch` (-20–+20 semitones), `volume_gain_db`, `sample_rate_hertz`, and `effects_profile_id` (device-tuned EQ presets like `telephony-class-application`).

## 3. Key Concepts

- **`SynthesisInput`.** What to synthesize: exactly one of `text=` or `ssml=`. SSML (`<speak>…</speak>`) unlocks `<break>`, `<emphasis>`, `<say-as interpret-as="date|cardinal|telephone">`, `<phoneme>`, `<prosody>`, and `<mark>` timepoints. **All input characters bill — SSML markup included.**
- **Voice tiers (the price/quality ladder).** **`Standard`** (cheapest, concatenative-ish, fine for IVR), **`WaveNet`** (neural, much more natural), **`Neural2`** (improved neural, the everyday workhorse), **`Studio`** (long-form narration, premium), and **`Chirp 3: HD`** (newest LLM-based generative voices, most natural/expressive). Voice `name` encodes the tier, e.g. `en-US-Neural2-F`.
- **`VoiceSelectionParams`.** `language_code` (BCP-47, e.g. `en-US`, `ja-JP`), optional `name` (specific voice), and `ssml_gender` (`MALE`/`FEMALE`/`NEUTRAL`) used only when `name` is omitted. `voices.list` enumerates everything available per language.
- **`AudioConfig`.** Output shaping: `audio_encoding` (`MP3`, `LINEAR16` = WAV PCM, `OGG_OPUS`, `MULAW`/`ALAW` for telephony), `speaking_rate`, `pitch`, `volume_gain_db`, `sample_rate_hertz`, and `effects_profile_id` (audio-profile EQ for phone/headphone/speaker).
- **Auth (ADC).** **Application Default Credentials**: set `GOOGLE_APPLICATION_CREDENTIALS` to a service-account JSON key, or `gcloud auth application-default login` for local dev, or use the attached service account on GCP. An API key also works for the REST endpoint. The Cloud TTS API must be **enabled** on the project and billing turned on.
- **Limits & async.** Synchronous `synthesize_speech` accepts ≤ **5000 bytes** of input. For books/long docs use the **Long Audio API** (`synthesize_long_audio`) — asynchronous, LINEAR16 only, output written to a GCS bucket. Default per-minute character quotas apply (raisable).
- **Billing.** Metered per **character per month**, by tier, with monthly free allowances. Roughly: **Standard ~\$4/1M chars (4M free)**, **WaveNet/Neural2 ~\$16/1M (1M free)**, **Studio ~\$160/1M (100K free)**, **Chirp 3 HD ~\$30/1M**. Numbers change — always check the pricing page.

## 4. Setup

```bash
pip install google-cloud-texttospeech       # official Python client (gRPC under the hood)

# Auth via Application Default Credentials (pick one):
export GOOGLE_APPLICATION_CREDENTIALS="/path/to/service-account.json"   # service account key
# ...or, for local dev:
gcloud auth application-default login

# In the Cloud Console: create a project, enable the "Cloud Text-to-Speech API", and turn on billing.
```

Get credentials from the [GCP Console](https://console.cloud.google.com/apis/credentials) → *Service Accounts* (grant the **Cloud Text-to-Speech User** role), then download a JSON key. Nothing runs locally — all synthesis happens on Google's servers, so every real call needs network + credentials and consumes your character quota.

The runnable cells below stay **self-contained and key-free**: Example 1 reproduces the **per-character cost math** across voice tiers, Example 2 builds the exact **request objects + REST body** you'd send, and Example 3 makes a **real synthesis call** gated behind `GOOGLE_APPLICATION_CREDENTIALS` — so the notebook always executes top-to-bottom.

In [ ]:
import sys, json

print(f"python {sys.version.split()[0]}")
print("Cloud TTS = hosted synthesis: input (text/ssml) + voice (lang+name=tier) + audio_config.")
print("The next two cells run with NO credentials and NO network: cost math, then the request shape.")

## 5. Worked Examples

### Example 1 — Cost accounting: estimate spend before you synthesize

Cloud TTS bills **per character of input per month**, and the rate depends entirely on the voice tier — Standard is ~4× cheaper than WaveNet/Neural2, and Studio is ~10× more again. Each tier also has a separate monthly **free allowance**. Before pushing a script you want to know which tier fits your free quota and what the overage costs. Pure Python, no credentials. (Rates change — treat these as ballpark and confirm on the pricing page.)

In [ ]:
# Approx. USD price per 1,000,000 characters, and free chars/month, by tier (verify on pricing page).
TIER = {
    "Standard":    {"per_1m": 4.0,   "free": 4_000_000},
    "WaveNet":     {"per_1m": 16.0,  "free": 1_000_000},
    "Neural2":     {"per_1m": 16.0,  "free": 1_000_000},
    "Studio":      {"per_1m": 160.0, "free": 100_000},
    "Chirp3-HD":   {"per_1m": 30.0,  "free": 0},
}

def monthly_cost(chars, tier):
    """Billable cost after the tier's free monthly allowance."""
    billable = max(0, chars - TIER[tier]["free"])
    return billable / 1_000_000 * TIER[tier]["per_1m"]

# A modest product: 200k chars/month of synthesis (say, ~40k short UI/IVR prompts).
VOLUME = 200_000
print(f"monthly volume: {VOLUME:,} characters\n")
print(f"{'tier':<12}{'$/1M':>8}{'free/mo':>14}{'cost @200k':>14}")
for tier, t in TIER.items():
    print(f"{tier:<12}{t['per_1m']:>8.0f}{t['free']:>14,}{monthly_cost(VOLUME, tier):>14.2f}")

# How big can a script get while staying inside each tier's free allowance?
print("\nlargest free-tier monthly volume (chars):")
for tier, t in TIER.items():
    print(f"  {tier:<12} {t['free']:>10,}")

# Reminder: SSML markup counts toward the character bill.
ssml = "<speak>Hello <break time='300ms'/> world.</speak>"
plain = "Hello world."
print(f"\nbilled chars  plain='{plain}' -> {len(plain)}")
print(f"billed chars  ssml -> {len(ssml)}  (every tag character is billed!)")

### Example 2 — Build the request: the three objects + the raw REST body

The Python client is a thin wrapper over one call that bundles three objects: `SynthesisInput`, `VoiceSelectionParams`, and `AudioConfig`. Knowing the raw contract lets you debug, port to any language, or hit the REST endpoint with plain `requests`. We assemble the equivalent JSON body and the `curl` you'd send — no network call is made, so nothing is synthesized and no quota is spent.

In [ ]:
# The three pieces of every synthesize request, as the REST JSON the client sends under the hood.
request_body = {
    "input": {
        # exactly one of these:
        "text": "Google Cloud turns this sentence into natural speech.",
        # "ssml": "<speak>Now <emphasis>this</emphasis> part is emphasized.</speak>",
    },
    "voice": {
        "languageCode": "en-US",
        "name": "en-US-Neural2-F",   # name encodes the tier: Standard/Wavenet/Neural2/Studio/Chirp3-HD
        # "ssmlGender": "FEMALE",    # only used when 'name' is omitted
    },
    "audioConfig": {
        "audioEncoding": "MP3",      # or LINEAR16 (WAV), OGG_OPUS, MULAW/ALAW (telephony)
        "speakingRate": 1.0,         # 0.25 - 4.0
        "pitch": 0.0,                # -20.0 - +20.0 semitones
        "volumeGainDb": 0.0,
        "effectsProfileId": ["telephony-class-application"],  # device-tuned EQ preset
    },
}

URL = "https://texttospeech.googleapis.com/v1/text:synthesize"
print("POST", URL)
print("  Authorization: Bearer $(gcloud auth print-access-token)   # or ?key=API_KEY")
print("  Content-Type: application/json\n")
print("body:")
print(json.dumps(request_body, indent=2))

print("\n# equivalent curl (response is base64 audio in JSON -> decode to out.mp3):")
print(f"""curl -s -X POST '{URL}' \\
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \\
  -H 'Content-Type: application/json' \\
  -d '{json.dumps(request_body)}' \\
  | jq -r '.audioContent' | base64 --decode > out.mp3""")

### Example 3 — A real synthesis call with the Python client (gated)

With `google-cloud-texttospeech` installed and `GOOGLE_APPLICATION_CREDENTIALS` set (and the API enabled on a billing-enabled project), synthesis is a few lines. This makes a real network call and **consumes quota**, so it's gated behind the env var. Either way the cell prints the canonical call shapes — text synthesis to bytes, SSML, and listing the voices available for a language.

In [ ]:
import os

if os.getenv("GOOGLE_APPLICATION_CREDENTIALS"):
    from google.cloud import texttospeech as tts

    client = tts.TextToSpeechClient()

    # 1) Plain text -> MP3 bytes -> file.
    resp = client.synthesize_speech(
        input=tts.SynthesisInput(text="Hello from Google Cloud Text-to-Speech."),
        voice=tts.VoiceSelectionParams(language_code="en-US", name="en-US-Neural2-F"),
        audio_config=tts.AudioConfig(audio_encoding=tts.AudioEncoding.MP3),
    )
    with open("out.mp3", "wb") as f:
        f.write(resp.audio_content)
    print(f"wrote out.mp3 ({len(resp.audio_content)} bytes)")

    # 2) List a few voices available for en-US.
    voices = client.list_voices(language_code="en-US").voices
    print("sample en-US voices:", [v.name for v in voices[:5]])
else:
    print("Set GOOGLE_APPLICATION_CREDENTIALS (and `pip install google-cloud-texttospeech`) to synthesize for real.\n")
    print("from google.cloud import texttospeech as tts")
    print("client = tts.TextToSpeechClient()\n")
    print("# text -> bytes")
    print("resp = client.synthesize_speech(")
    print('    input=tts.SynthesisInput(text="Hello."),')
    print('    voice=tts.VoiceSelectionParams(language_code="en-US", name="en-US-Neural2-F"),')
    print("    audio_config=tts.AudioConfig(audio_encoding=tts.AudioEncoding.MP3))")
    print('open("out.mp3", "wb").write(resp.audio_content)\n')
    print("# SSML instead of text (tags are billed):")
    print('tts.SynthesisInput(ssml="<speak>Wait <break time=\'500ms\'/> then go.</speak>")\n')
    print("# enumerate voices for a language:")
    print('client.list_voices(language_code="en-US").voices')

## 6. Gotchas & Pitfalls

- **The 5000-byte synchronous limit.** `synthesize_speech` rejects input over 5000 bytes (UTF-8, **SSML tags included**). Long documents must be chunked client-side or sent through the async **Long Audio API** (`synthesize_long_audio`), which is LINEAR16-only and writes to a GCS bucket. Plan for it before you ship a book reader.
- **SSML markup is billed.** Every character of input counts toward your bill — `<speak>`, `<break>`, attributes and all. Verbose SSML can multiply your character cost; keep markup tight.
- **Voice name *is* the price tier.** `en-US-Standard-C` and `en-US-Studio-O` differ ~40× in price. Picking a fancy voice by habit can blow the budget; choose the cheapest tier that sounds good enough, and confirm the `name` actually exists for the `language_code` (mismatches throw `INVALID_ARGUMENT`).
- **`pitch` is in semitones, `speakingRate` is a multiplier.** `pitch` ranges -20…+20 *semitones* (not Hz, not a 0–1 knob) and `speakingRate` is 0.25–4.0× normal. Confusing the two yields chipmunk or molasses audio. Some newer voices (Studio/Chirp) ignore or restrict these.
- **Auth/enable footguns.** `DefaultCredentialsError` means ADC isn't set (`GOOGLE_APPLICATION_CREDENTIALS` or `gcloud auth application-default login`). `PERMISSION_DENIED`/403 usually means the **Cloud Text-to-Speech API isn't enabled** on the project or the service account lacks the role, and synthesis silently requires **billing enabled** even within the free allowance.
- **Encoding must match downstream.** Telephony wants `MULAW`/`ALAW` at 8 kHz; further DSP wants `LINEAR16` (raw WAV PCM); web delivery wants `MP3`/`OGG_OPUS`. Mismatched `sample_rate_hertz` for the voice/encoding causes resampling artifacts or errors.
- **No general voice cloning.** Unlike ElevenLabs/Coqui, you can't clone an arbitrary voice from a sample — you choose from Google's catalog (Custom Voice is a gated enterprise program). If cloning is the requirement, this isn't your tool.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs Cloud TTS |
|---|---|---|
| **Google Cloud TTS** | GCP-native apps, ~50 languages, SSML control, telephony profiles, enterprise IAM/compliance | Less expressive than ElevenLabs; no general voice cloning; per-char cost; cloud-only |
| **ElevenLabs** | Best-in-class naturalness, instant voice cloning, expressive/emotive output | Pricier at quality tier; not GCP-native; fewer enterprise controls |
| **Amazon Polly** | AWS-native apps, neural voices, newscaster styles, generative voices | Tied to AWS; voice catalog/quality comparable, ecosystem-dependent |
| **Azure AI Speech** | Azure-native, huge voice catalog, custom neural voice, rich SSML/styles | Tied to Azure; setup heavier; comparable quality |
| **OpenAI TTS** | Simple, cheap, decent quality inside the OpenAI stack | Few voices, no SSML/voice cloning, less delivery control |
| **Coqui / Piper (self-hosted)** | Offline/on-device, no per-use bill, data never leaves the box | You run the ops; quality trails hosted; no managed scaling |

**Rule of thumb:** choose **Google Cloud TTS when you're on GCP (or want its IAM/compliance/telephony plumbing), need broad language coverage with SSML control, and absolute top-tier naturalness isn't the deciding factor.** Reach for **ElevenLabs** when expressiveness or voice cloning matters most, for **Polly/Azure** to stay native in those clouds, for **OpenAI TTS** when you just want cheap decent voice in that stack, and for **Coqui/Piper** when audio must stay offline or you're optimizing high-volume cost.

## 8. Resources

- **Cloud TTS docs (overview + how-tos)** — https://cloud.google.com/text-to-speech/docs
- **Supported voices & languages** — the full catalog with tier and `name` per language: https://cloud.google.com/text-to-speech/docs/voices
- **SSML reference** — every tag (`break`, `say-as`, `prosody`, `phoneme`, `mark`): https://cloud.google.com/text-to-speech/docs/ssml
- **REST API reference (`text:synthesize`)** — request/response schema: https://cloud.google.com/text-to-speech/docs/reference/rest/v1/text/synthesize
- **Long Audio API** — async synthesis of long-form content to GCS: https://cloud.google.com/text-to-speech/docs/create-audio-text-long-audio-synthesis
- **Pricing** — per-tier rates and monthly free allowances: https://cloud.google.com/text-to-speech/pricing
- **Python client (`google-cloud-texttospeech`)** — install, samples, API: https://cloud.google.com/python/docs/reference/texttospeech/latest

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
TIER = {}
MAX_REQUEST_BYTES = 5000


def tier_of(voice_name):
    ...


def build_request(language_code, **kwargs):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE